In [8]:
# Librerias necesarias
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt 
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

In [9]:
# Cargar df 
df = pd.read_csv("datos/SIFT_dis_sis.csv")

In [10]:
df

,Tipo,SRA,Linaje,Sublinaje,Familia,Pais,SIFT_PREDICTION,GENE_ID,POS,REF_ALLELE,...,GENE_NAME,VARIANT_TYPE,REF_AMINO,ALT_AMINO,AMINO_POS,SIFT_SCORE,SIFT_MEDIAN,NUM_SEQS,Drtype,Mutacion
0,Discrepante,ERR2510876,L2,L2.2.1,Beijing,Italy,TOLERATED,Rv0191,222925,G,...,NaN,NONSYNONYMOUS,A,T,213.0,0.452,3.45,8.0,MDR-TB,Rv0191 A213T
1,Discrepante,ERR2510876,L2,L2.2.1,Beijing,Italy,TOLERATED,Rv0194,227098,T,...,NaN,NONSYNONYMOUS,M,T,74.0,1.000,2.92,49.0,MDR-TB,Rv0194 M74T
2,Discrepante,ERR2510876,L2,L2.2.1,Beijing,Italy,DELETERIOUS,Rv0194,230170,C,...,NaN,NONSYNONYMOUS,P,L,1098.0,0.002,2.91,60.0,MDR-TB,Rv0194 P1098L
3,Discrepante,ERR2510876,L2,L2.2.1,Beijing,Italy,TOLERATED,Rv0450c,541201,A,...,mmpL4,SYNONYMOUS,L,L,97.0,1.000,2.70,113.0,MDR-TB,mmpL4 L97L
4,Discrepante,ERR2510876,L2,L2.2.1,Beijing,Italy,TOLERATED,Rv0507,597816,A,...,mmpL2,SYNONYMOUS,A,A,206.0,0.482,2.68,112.0,MDR-TB,mmpL2 A206A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29511,Susceptible,ERR400454,L3,L3,CAS1-Delhi,NaN,TOLERATED,Rv2688c,3005185,G,...,NaN,NONSYNONYMOUS,P,T,156.0,0.839,2.70,54.0,NaN,Rv2688c P156T
29512,Susceptible,ERR400454,L3,L3,CAS1-Delhi,NaN,TOLERATED,Rv2936,3273107,C,...,drrA,SYNONYMOUS,A,A,298.0,0.786,2.61,91.0,NaN,drrA A298A
29513,Susceptible,ERR400454,L3,L3,CAS1-Delhi,NaN,TOLERATED,Rv3239c,3614982,T,...,NaN,SYNONYMOUS,L,L,874.0,1.000,3.50,19.0,NaN,Rv3239c L874L
29514,Susceptible,ERR400454,L3,L3,CAS1-Delhi,NaN,TOLERATED,Rv3728,4175354,A,...,NaN,NONSYNONYMOUS,E,A,161.0,1.000,3.66,8.0,NaN,Rv3728 E161A


In [11]:
# =========================
# 1. Crear columna Mutacion
# =========================
df["Mutacion"] = (
    df["GENE_NAME"].fillna(df["GENE_ID"]).astype(str) + " " +
    df["REF_AMINO"].fillna(df["REF_ALLELE"]).astype(str) +
    df["AMINO_POS"].fillna(df["POS"]).astype("Int64").astype(str) +
    df["ALT_AMINO"].fillna(df["ALT_ALLELE"]).astype(str)
)

# ==========================================
# 2. Tabla de contingencia Mutacion vs Tipo
# ==========================================
ct = pd.crosstab(df["Mutacion"], df["Tipo"])
ct = ct.reindex(columns=["Discrepante", "Susceptible"], fill_value=0)

# Totales globales
total_disc = ct["Discrepante"].sum()
total_susc = ct["Susceptible"].sum()

# =========================
# 3. Análisis por mutación
# =========================
rows = []

for mut, r in ct.iterrows():
    a = int(r["Discrepante"])   # Mutación en discrepantes
    b = int(r["Susceptible"])   # Mutación en susceptibles
    c = int(total_disc - a)     # No mutación en discrepantes
    d = int(total_susc - b)     # No mutación en susceptibles

    tabla_2x2 = [[a, b], [c, d]]

    # OR con corrección de Haldane-Anscombe
    or_corr = ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))

    # Elegir prueba estadística
    if min(a, b, c, d) == 0:
        _, p = fisher_exact(tabla_2x2)
        test = "Fisher"
    else:
        chi2, p, dof, expected = chi2_contingency(tabla_2x2, correction=False)

        if expected.min() < 5:
            _, p = fisher_exact(tabla_2x2)
            test = "Fisher"
        else:
            test = "Chi2"

    rows.append({
        "Mutacion": mut,
        "a_disc": a,
        "b_susc": b,
        "c_no_mut_disc": c,
        "d_no_mut_susc": d,
        "OR": or_corr,
        "p": p,
        "test": test
    })

# =========================
# 4. Crear DataFrame final
# =========================
res = pd.DataFrame(rows)

# Ajuste de p-values por FDR (Benjamini-Hochberg)
reject, pvals_corr, _, _ = multipletests(res["p"], method="fdr_bh")

res["p_adj"] = pvals_corr
res["significativo"] = reject

# Ordenar por significancia
res = res.sort_values("p_adj", ascending=True)

# ==========================================
# 5. Filtrar mutaciones significativas
# ==========================================
sig = res[(res["p_adj"] < 0.05) & (res["OR"] > 1)].copy()

# Ver resultados
print("Resultados completos:")
print(res.head())

print("\nMutaciones significativas:")
print(sig.head())

Resultados completos:
                Mutacion  a_disc  b_susc  c_no_mut_disc  d_no_mut_susc  \
60          Rv0191 A213T     291      88          15280          13857   
157        Rv0194 P1098L     292      89          15279          13856   
692        Rv1458c T133A     291      88          15280          13857   
2053         mmpL8 L465L     290      89          15281          13856   
487   Rv1258c T1406760TG     279      83          15292          13862   

            OR             p  test         p_adj  significativo  
60    2.987051  4.104363e-21  Chi2  4.074192e-18           True  
157   2.963789  5.468714e-21  Chi2  4.074192e-18           True  
692   2.987051  4.104363e-21  Chi2  4.074192e-18           True  
2053  2.943138  1.095746e-20  Chi2  4.140857e-18           True  
487   3.034299  1.111639e-20  Chi2  4.140857e-18           True  

Mutaciones significativas:
                Mutacion  a_disc  b_susc  c_no_mut_disc  d_no_mut_susc  \
60          Rv0191 A213T     291   

In [12]:
sig

,Mutacion,a_disc,b_susc,c_no_mut_disc,d_no_mut_susc,OR,p,test,p_adj,significativo
60,Rv0191 A213T,291,88,15280,13857,2.987051,4.104363e-21,Chi2,4.074192e-18,True
157,Rv0194 P1098L,292,89,15279,13856,2.963789,5.468714e-21,Chi2,4.074192e-18,True
692,Rv1458c T133A,291,88,15280,13857,2.987051,4.104363e-21,Chi2,4.074192e-18,True
2053,mmpL8 L465L,290,89,15281,13856,2.943138,1.095746e-20,Chi2,4.140857e-18,True
487,Rv1258c T1406760TG,279,83,15292,13862,3.034299,1.111639e-20,Chi2,4.140857e-18,True
296,Rv1217c A173T,289,88,15282,13857,2.966168,8.245355e-21,Chi2,4.140857e-18,True
1829,mmpL5 D767N,263,75,15308,13870,3.162228,1.689092e-20,Chi2,5.393028e-18,True
341,Rv1217c S204S,386,169,15185,13776,2.068662,1.234669e-15,Chi2,3.449356e-13,True
1719,mmpL2 R890R,436,242,15135,13703,1.629698,1.089822e-09,Chi2,2.435752e-07,True
1896,mmpL5 T794I,435,242,15136,13703,1.625857,1.333849e-09,Chi2,2.710139e-07,True


# Asociar mutaciones a linajes

In [13]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

# === 1) Dejar una sola vez cada combinación cepa-mutación ===
df = df.drop_duplicates(subset=["SRA", "Linaje", "Mutacion"]).copy()

# === 2) Quedarnos con linajes principales ===
main_lineages = ["L1", "L2", "L3", "L4"]
df = df[df["Linaje"].isin(main_lineages)].copy()

# === 3) Tabla global Linaje ===
totals = (
    df[["SRA", "Linaje"]]
    .drop_duplicates()["Linaje"]
    .value_counts()
    .reindex(main_lineages, fill_value=0)
)

print("Total de cepas por linaje:")
print(totals)

# === 4) Conteo de mutaciones por linaje ===
mut_lin = (
    df[["SRA", "Linaje", "Mutacion"]]
    .drop_duplicates()
    .groupby(["Mutacion", "Linaje"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=main_lineages, fill_value=0)
)

# === 5) Chi-cuadrada por mutación ===
results = []

for mut, row in mut_lin.iterrows():
    present = row.values.astype(int)

    # Saltar mutaciones muy raras
    if present.sum() < 5:
        continue

    absent = totals.values - present
    table = np.vstack([present, absent]).T  # linajes x (presente/ausente)

    chi2, p, dof, exp = chi2_contingency(table)

    results.append({
        "Mutacion": mut,
        "n_cepas_con_mutacion": int(present.sum()),
        "L1": int(present[0]),
        "L2": int(present[1]),
        "L3": int(present[2]),
        "L4": int(present[3]),
        "chi2": chi2,
        "p": p,
        "min_expected": float(exp.min()),
        "cells_expected_lt5": int((exp < 5).sum())
    })

results = pd.DataFrame(results)

# === 6) Ajuste por comparaciones múltiples ===
if not results.empty:
    results["p_adj_bh"] = multipletests(results["p"], method="fdr_bh")[1]
else:
    results["p_adj_bh"] = pd.Series(dtype=float)

results = results.sort_values(["p_adj_bh", "p"])

# === 7) Resultados confiables y significativos ===
valid = results[results["min_expected"] >= 5].copy()
sig = valid[valid["p_adj_bh"] < 0.05].copy()

print("\nTop mutaciones válidas (expected >= 5):")
print(valid.head(20)[[
    "Mutacion", "n_cepas_con_mutacion",
    "L1", "L2", "L3", "L4", "p", "p_adj_bh"
]])

print("\nTop mutaciones significativas:")
print(sig.head(20)[[
    "Mutacion", "n_cepas_con_mutacion",
    "L1", "L2", "L3", "L4", "p", "p_adj_bh"
]])

Total de cepas por linaje:
Linaje
L1     100
L2     374
L3     172
L4    1120
Name: count, dtype: int64

Top mutaciones válidas (expected >= 5):
                Mutacion  n_cepas_con_mutacion   L1   L2   L3  L4    p  \
6           Rv0191 A213T                   371    0  371    0   0  0.0   
16         Rv0194 P1098L                   373    0  373    0   0  0.0   
27           Rv0842 L45L                   100  100    0    0   0  0.0   
34         Rv1217c A173T                   369    0  369    0   0  0.0   
40         Rv1217c S204S                   541    0  370  170   1  0.0   
53    Rv1258c T1406760TG                   356    0  356    0   0  0.0   
56   Rv1273c CC1422666TT                   162    0    0  162   0  0.0   
67         Rv1458c T133A                   372    0  372    0   0  0.0   
90           Rv2265 G32S                   100  100    0    0   0  0.0   
120          Rv2994 W68*                   100  100    0    0   0  0.0   
150           bacA I603V                 